# AD merged GEO cohorts — download, preprocess, merge

Builds one expression matrix from five post-mortem AD brain cohorts, entirely
inside Kaggle. Every file comes straight from NCBI GEO / NCBI Gene; nothing is
uploaded from a local machine.

| cohort | platform | kept |
|---|---|---|
| GSE48350 | Affymetrix U133 Plus 2.0 | the project's Kaggle datasets (`alzheimercsv`, `labels1`) |
| GSE33000 | Agilent/Rosetta 44K (log10 ratio vs pool) | AD + non-demented; Huntington's dropped |
| GSE132903 | Illumina HT-12 v4 | AD + non-demented |
| GSE118553 | Illumina HT-12 v4 | AD + control; asymptomatic AD and cerebellum dropped |
| GSE84422 | Affymetrix U133A (GPL96 arrays) | definite/probable AD + normal; possible AD, caudate and putamen dropped |

**Preprocessing — the project's own recipe, applied to every cohort:** probes
mapped to gene symbols, unmapped and multi-gene probes discarded, several probes
of one symbol collapsed by their arithmetic mean. Because the platforms differ,
symbols are first normalised through NCBI Gene (Entrez ID → current official
symbol), so the same gene carries the same name in every cohort.

**Then:** donors aged ≥ 60 only (as in V6ADbio); genes kept only if measured in
all five cohorts; and each gene **z-scored within each cohort**. The last step
is needed because the cohorts are on different scales (log2 intensities vs
log10 ratios). It uses no labels, so it cannot leak diagnosis.

Output (`/kaggle/working`): `merged_expression.csv.gz`, `merged_labels.csv`,
`merged_metadata.csv`, `cohort_table.csv`, `gene_coverage.csv`.


In [ ]:
# ======================================================================
# Downloads and parsers
# ======================================================================
import os, re, io, gzip, glob, requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT, TMP = "/kaggle/working", "/kaggle/tmp"
os.makedirs(TMP, exist_ok=True)
AGE_MIN = 60
HDR = {"User-Agent": "ad-merged-geo-cohorts (kaggle)"}


def fetch(url, name):
    """Stream a file to Kaggle's scratch disk, with retries."""
    path = f"{TMP}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for attempt in range(5):
        try:
            with requests.get(url, headers=HDR, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            print(f"downloaded {name}: {os.path.getsize(path)/1e6:.1f} MB")
            return path
        except Exception as e:                       # noqa: BLE001
            print(f"  retry {attempt+1} for {name}: {e}")
    raise RuntimeError(f"could not download {url}")


def series_matrix(acc, suffix=""):
    stub = acc[:-3] + "nnn"
    fn = f"{acc}{suffix}_series_matrix.txt.gz"
    return fetch(f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{acc}/matrix/{fn}", fn)


def read_series_matrix(path):
    """-> (expression DataFrame probes x GSM, per-sample metadata DataFrame)."""
    meta_lines, table = [], []
    with gzip.open(path, "rt", errors="ignore") as f:
        in_table = False
        for line in f:
            if line.startswith("!series_matrix_table_begin"):
                in_table = True; continue
            if line.startswith("!series_matrix_table_end"):
                break
            (table if in_table else meta_lines).append(line)
    expr = pd.read_csv(io.StringIO("".join(table)), sep="\t", index_col=0)
    expr.index = expr.index.astype(str)
    expr = expr.apply(pd.to_numeric, errors="coerce")

    gsm = None; rows = {}
    for line in meta_lines:
        parts = [p.strip().strip('"') for p in line.rstrip("\n").split("\t")]
        key, vals = parts[0], parts[1:]
        if key == "!Sample_geo_accession":
            gsm = vals
        elif key in ("!Sample_title", "!Sample_source_name_ch1"):
            rows[key[8:]] = vals
        elif key.startswith("!Sample_characteristics_ch"):
            for i, v in enumerate(vals):
                if ":" in v:
                    k, val = v.split(":", 1)
                    rows.setdefault(k.strip().lower(), [None] * len(vals))[i] = val.strip()
    meta = pd.DataFrame(rows, index=gsm)
    meta.index.name = "gsm"
    return expr, meta


def platform_table(gpl):
    path = fetch(f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gpl}&targ=self&form=text&view=data",
                 f"{gpl}.txt")
    lines, on = [], False
    with open(path, errors="ignore") as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                on = True; continue
            if line.startswith("!platform_table_end"):
                break
            if on:
                lines.append(line)
    t = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
    t["ID"] = t["ID"].astype(str)
    return t.set_index("ID")


def maybe_log2(df, name):
    """GEO2R's rule for whether a matrix still needs log2."""
    qx = np.nanquantile(df.values.astype(float), [0, .25, .5, .75, .99, 1.0])
    need = (qx[4] > 100) or (qx[5] - qx[0] > 50 and qx[1] > 0)
    print(f"{name}: quantiles {np.round(qx, 2)} -> {'log2 applied' if need else 'already log scale'}")
    if need:
        df = np.log2(df.clip(lower=0) + 1)
    return df

# ---- NCBI Gene: Entrez ID -> current official symbol; symbol/synonym -> Entrez
gi = pd.read_csv(fetch("https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz",
                       "Homo_sapiens.gene_info.gz"), sep="\t", dtype=str,
                 usecols=["GeneID", "Symbol", "Synonyms"])
ENTREZ2SYM = dict(zip(gi.GeneID, gi.Symbol))
OFFICIAL = set(gi.Symbol)
_syn = {}
for gid, syns in zip(gi.GeneID, gi.Synonyms):
    if isinstance(syns, str) and syns != "-":
        for s in syns.split("|"):
            _syn.setdefault(s, set()).add(gid)
SYN2SYM = {s: ENTREZ2SYM[next(iter(g))] for s, g in _syn.items() if len(g) == 1 and s not in OFFICIAL}
print(f"NCBI Gene: {len(OFFICIAL):,} official symbols, {len(SYN2SYM):,} unambiguous synonyms")


def norm_symbol(s):
    if not isinstance(s, str) or not s or s == "-":
        return None
    s = s.strip()
    if s in OFFICIAL:
        return s
    return SYN2SYM.get(s)


def probes_to_symbols(expr, ann, entrez_col=None, symbol_col=None, name=""):
    """Project recipe: probe -> one gene; unmapped / multi-gene probes dropped;
    probes of the same gene averaged."""
    sym = pd.Series(index=ann.index, dtype=object)
    if entrez_col is not None:
        ez = ann[entrez_col].fillna("")
        single = ~ez.str.contains("///") & ez.str.match(r"^\d+$")
        sym[single] = ez[single].map(ENTREZ2SYM)
    if symbol_col is not None:
        sy = ann[symbol_col].fillna("")
        fill = sym.isna() & ~sy.str.contains("///") & (sy != "")
        sym[fill] = sy[fill].map(norm_symbol)
    sym = sym.reindex(expr.index)
    keep = sym.notna()
    print(f"{name}: {len(expr):,} probes -> {keep.sum():,} single-gene probes -> "
          f"{sym[keep].nunique():,} genes")
    return expr[keep].groupby(sym[keep]).mean()

print("parsers ready")


In [ ]:
# ======================================================================
# GSE48350 — the project's own Kaggle datasets
# ======================================================================
GSE48350_META = [
    ('GSM300166', 'C10', 'postcentral gyrus', 91, 'female'),
    ('GSM300167', 'C10', 'superior frontal gyrus', 91, 'female'),
    ('GSM300168', 'C105', 'hippocampus', 96, 'female'),
    ('GSM300169', 'C106', 'hippocampus', 82, 'male'),
    ('GSM300170', 'C108', 'hippocampus', 84, 'male'),
    ('GSM300171', 'C109', 'hippocampus', 87, 'female'),
    ('GSM300172', 'C111', 'hippocampus', 82, 'female'),
    ('GSM300173', 'C12', 'entorhinal cortex', 45, 'male'),
    ('GSM300174', 'C12', 'hippocampus', 45, 'male'),
    ('GSM300175', 'C12', 'postcentral gyrus', 45, 'male'),
    ('GSM300176', 'C12', 'superior frontal gyrus', 45, 'male'),
    ('GSM300177', 'C14', 'entorhinal cortex', 95, 'male'),
    ('GSM300178', 'C14', 'hippocampus', 95, 'male'),
    ('GSM300179', 'C14', 'postcentral gyrus', 95, 'male'),
    ('GSM300180', 'C14', 'superior frontal gyrus', 95, 'male'),
    ('GSM300181', 'C15', 'entorhinal cortex', 80, 'male'),
    ('GSM300182', 'C15', 'hippocampus', 80, 'male'),
    ('GSM300183', 'C15', 'postcentral gyrus', 80, 'male'),
    ('GSM300185', 'C16-07', 'hippocampus', 91, 'male'),
    ('GSM300186', 'C17', 'entorhinal cortex', 45, 'female'),
    ('GSM300187', 'C17', 'hippocampus', 45, 'female'),
    ('GSM300188', 'C17', 'superior frontal gyrus', 45, 'female'),
    ('GSM300189', 'C18', 'entorhinal cortex', 74, 'female'),
    ('GSM300190', 'C18', 'hippocampus', 74, 'female'),
    ('GSM300191', 'C18', 'superior frontal gyrus', 74, 'female'),
    ('GSM300192', 'C2', 'entorhinal cortex', 99, 'female'),
    ('GSM300193', 'C2', 'hippocampus', 99, 'female'),
    ('GSM300194', 'C2', 'postcentral gyrus', 99, 'female'),
    ('GSM300195', 'C2', 'superior frontal gyrus', 99, 'female'),
    ('GSM300196', 'C21', 'entorhinal cortex', 74, 'female'),
    ('GSM300197', 'C21', 'hippocampus', 74, 'female'),
    ('GSM300198', 'C21', 'postcentral gyrus', 74, 'female'),
    ('GSM300199', 'C21', 'superior frontal gyrus', 74, 'female'),
    ('GSM300200', 'C26', 'postcentral gyrus', 92, 'female'),
    ('GSM300201', 'C26', 'superior frontal gyrus', 92, 'female'),
    ('GSM300202', 'C27', 'postcentral gyrus', 91, 'female'),
    ('GSM300203', 'C27', 'superior frontal gyrus', 91, 'female'),
    ('GSM300204', 'C28', 'entorhinal cortex', 83, 'male'),
    ('GSM300205', 'C28', 'hippocampus', 83, 'male'),
    ('GSM300206', 'C28', 'postcentral gyrus', 83, 'male'),
    ('GSM300207', 'C28', 'superior frontal gyrus', 83, 'male'),
    ('GSM300208', 'C29', 'entorhinal cortex', 28, 'male'),
    ('GSM300209', 'C29', 'hippocampus', 28, 'male'),
    ('GSM300210', 'C29', 'postcentral gyrus', 28, 'male'),
    ('GSM300211', 'C29', 'superior frontal gyrus', 28, 'male'),
    ('GSM300212', 'C30', 'postcentral gyrus', 74, 'female'),
    ('GSM300213', 'C30', 'superior frontal gyrus', 74, 'female'),
    ('GSM300214', 'C34', 'entorhinal cortex', 97, 'male'),
    ('GSM300215', 'C34', 'hippocampus', 97, 'male'),
    ('GSM300216', 'C34', 'postcentral gyrus', 97, 'male'),
    ('GSM300217', 'C34', 'superior frontal gyrus', 97, 'male'),
    ('GSM300218', 'C35', 'entorhinal cortex', 34, 'female'),
    ('GSM300219', 'C35', 'hippocampus', 34, 'female'),
    ('GSM300220', 'C35', 'postcentral gyrus', 34, 'female'),
    ('GSM300221', 'C35', 'superior frontal gyrus', 34, 'female'),
    ('GSM300222', 'C38', 'entorhinal cortex', 74, 'female'),
    ('GSM300223', 'C38', 'hippocampus', 74, 'female'),
    ('GSM300224', 'C38', 'postcentral gyrus', 74, 'female'),
    ('GSM300225', 'C38', 'superior frontal gyrus', 74, 'female'),
    ('GSM300226', 'C40', 'postcentral gyrus', 90, 'female'),
    ('GSM300227', 'C40-04', 'hippocampus', 91, 'female'),
    ('GSM300228', 'C42', 'entorhinal cortex', 44, 'female'),
    ('GSM300229', 'C42', 'superior frontal gyrus', 44, 'female'),
    ('GSM300230', 'C45', 'entorhinal cortex', 37, 'female'),
    ('GSM300231', 'C45', 'hippocampus', 37, 'female'),
    ('GSM300232', 'C45', 'postcentral gyrus', 37, 'female'),
    ('GSM300233', 'C45', 'superior frontal gyrus', 37, 'female'),
    ('GSM300234', 'C46', 'entorhinal cortex', 85, 'male'),
    ('GSM300235', 'C46', 'hippocampus', 85, 'male'),
    ('GSM300236', 'C46', 'postcentral gyrus', 85, 'male'),
    ('GSM300237', 'C46', 'superior frontal gyrus', 85, 'male'),
    ('GSM300238', 'C47', 'entorhinal cortex', 70, 'female'),
    ('GSM300239', 'C47', 'hippocampus', 70, 'female'),
    ('GSM300240', 'C47', 'postcentral gyrus', 70, 'female'),
    ('GSM300241', 'C47', 'superior frontal gyrus', 70, 'female'),
    ('GSM300242', 'C52', 'entorhinal cortex', 64, 'female'),
    ('GSM300243', 'C52', 'hippocampus', 64, 'female'),
    ('GSM300244', 'C52', 'postcentral gyrus', 64, 'female'),
    ('GSM300245', 'C52', 'superior frontal gyrus', 64, 'female'),
    ('GSM300246', 'C53', 'postcentral gyrus', 70, 'male'),
    ('GSM300247', 'C53', 'superior frontal gyrus', 70, 'male'),
    ('GSM300248', 'C54', 'entorhinal cortex', 85, 'female'),
    ('GSM300249', 'C54', 'postcentral gyrus', 85, 'female'),
    ('GSM300250', 'C54', 'superior frontal gyrus', 85, 'female'),
    ('GSM300251', 'C59', 'superior frontal gyrus', 91, 'female'),
    ('GSM300252', 'C66', 'entorhinal cortex', 21, 'male'),
    ('GSM300253', 'C66', 'postcentral gyrus', 21, 'male'),
    ('GSM300254', 'C66', 'superior frontal gyrus', 21, 'male'),
    ('GSM300255', 'C67', 'hippocampus', 69, 'male'),
    ('GSM300256', 'C67', 'postcentral gyrus', 69, 'male'),
    ('GSM300258', 'C68', 'entorhinal cortex', 40, 'male'),
    ('GSM300259', 'C68', 'postcentral gyrus', 40, 'male'),
    ('GSM300260', 'C68', 'superior frontal gyrus', 40, 'male'),
    ('GSM300261', 'C71', 'entorhinal cortex', 52, 'male'),
    ('GSM300262', 'C71', 'hippocampus', 52, 'male'),
    ('GSM300263', 'C71', 'postcentral gyrus', 52, 'male'),
    ('GSM300264', 'C71', 'superior frontal gyrus', 52, 'male'),
    ('GSM300265', 'C72', 'postcentral gyrus', 75, 'male'),
    ('GSM300266', 'C72', 'superior frontal gyrus', 75, 'male'),
    ('GSM300267', 'C73', 'entorhinal cortex', 86, 'male'),
    ('GSM300268', 'C73', 'hippocampus', 86, 'male'),
    ('GSM300269', 'C73', 'postcentral gyrus', 86, 'male'),
    ('GSM300270', 'C73', 'superior frontal gyrus', 86, 'male'),
    ('GSM300271', 'C76', 'entorhinal cortex', 26, 'female'),
    ('GSM300272', 'C76', 'hippocampus', 26, 'female'),
    ('GSM300273', 'C76', 'postcentral gyrus', 26, 'female'),
    ('GSM300274', 'C76', 'superior frontal gyrus', 26, 'female'),
    ('GSM300275', 'C77', 'entorhinal cortex', 20, 'male'),
    ('GSM300276', 'C77', 'hippocampus', 20, 'male'),
    ('GSM300277', 'C77', 'postcentral gyrus', 20, 'male'),
    ('GSM300278', 'C77', 'superior frontal gyrus', 20, 'male'),
    ('GSM300279', 'C78', 'entorhinal cortex', 20, 'male'),
    ('GSM300280', 'C78', 'hippocampus', 20, 'male'),
    ('GSM300281', 'C78', 'postcentral gyrus', 20, 'male'),
    ('GSM300282', 'C78', 'superior frontal gyrus', 20, 'male'),
    ('GSM300283', 'C79', 'postcentral gyrus', 36, 'female'),
    ('GSM300284', 'C79', 'superior frontal gyrus', 36, 'female'),
    ('GSM300285', 'C8', 'entorhinal cortex', 69, 'male'),
    ('GSM300286', 'C8', 'hippocampus', 69, 'male'),
    ('GSM300287', 'C8', 'postcentral gyrus', 69, 'male'),
    ('GSM300288', 'C8', 'superior frontal gyrus', 69, 'male'),
    ('GSM300289', 'C80', 'entorhinal cortex', 44, 'female'),
    ('GSM300290', 'C80', 'hippocampus', 44, 'female'),
    ('GSM300291', 'C80', 'postcentral gyrus', 44, 'female'),
    ('GSM300292', 'C80', 'superior frontal gyrus', 44, 'female'),
    ('GSM300293', 'C81', 'entorhinal cortex', 48, 'female'),
    ('GSM300294', 'C81', 'hippocampus', 48, 'female'),
    ('GSM300295', 'C81', 'postcentral gyrus', 48, 'female'),
    ('GSM300296', 'C81', 'superior frontal gyrus', 48, 'female'),
    ('GSM300297', 'C82', 'entorhinal cortex', 30, 'female'),
    ('GSM300298', 'C82', 'hippocampus', 30, 'female'),
    ('GSM300299', 'C82', 'superior frontal gyrus', 30, 'female'),
    ('GSM300300', 'C83', 'entorhinal cortex', 20, 'male'),
    ('GSM300301', 'C83', 'hippocampus', 20, 'male'),
    ('GSM300302', 'C83', 'postcentral gyrus', 20, 'male'),
    ('GSM300303', 'C83', 'superior frontal gyrus', 20, 'male'),
    ('GSM300304', 'C84', 'entorhinal cortex', 33, 'male'),
    ('GSM300305', 'C84', 'hippocampus', 33, 'male'),
    ('GSM300306', 'C84', 'postcentral gyrus', 33, 'male'),
    ('GSM300307', 'C84', 'superior frontal gyrus', 33, 'male'),
    ('GSM300308', 'C85', 'entorhinal cortex', 22, 'male'),
    ('GSM300309', 'C85', 'hippocampus', 22, 'male'),
    ('GSM300310', 'C85', 'postcentral gyrus', 22, 'male'),
    ('GSM300311', 'C85', 'superior frontal gyrus', 22, 'male'),
    ('GSM300312', 'C86', 'entorhinal cortex', 42, 'male'),
    ('GSM300313', 'C86', 'hippocampus', 42, 'male'),
    ('GSM300314', 'C86', 'postcentral gyrus', 42, 'male'),
    ('GSM300315', 'C86', 'superior frontal gyrus', 42, 'male'),
    ('GSM300316', 'C87', 'entorhinal cortex', 45, 'male'),
    ('GSM300317', 'C87', 'hippocampus', 45, 'male'),
    ('GSM300318', 'C87', 'postcentral gyrus', 45, 'male'),
    ('GSM300319', 'C87', 'superior frontal gyrus', 45, 'male'),
    ('GSM300320', 'C88', 'entorhinal cortex', 47, 'female'),
    ('GSM300321', 'C88', 'hippocampus', 47, 'female'),
    ('GSM300322', 'C88', 'postcentral gyrus', 47, 'female'),
    ('GSM300323', 'C88', 'superior frontal gyrus', 47, 'female'),
    ('GSM300324', 'C93', 'entorhinal cortex', 69, 'male'),
    ('GSM300325', 'C93', 'hippocampus', 69, 'male'),
    ('GSM300326', 'C93', 'postcentral gyrus', 69, 'male'),
    ('GSM300327', 'C93', 'superior frontal gyrus', 69, 'male'),
    ('GSM300328', 'C94', 'entorhinal cortex', 91, 'female'),
    ('GSM300329', 'C94', 'hippocampus', 91, 'female'),
    ('GSM300330', 'C94', 'postcentral gyrus', 91, 'female'),
    ('GSM300331', 'C94', 'superior frontal gyrus', 91, 'female'),
    ('GSM300332', 'C96', 'entorhinal cortex', 75, 'male'),
    ('GSM300333', 'C96', 'hippocampus', 75, 'male'),
    ('GSM300335', 'C96', 'superior frontal gyrus', 75, 'male'),
    ('GSM300338', 'C98', 'entorhinal cortex', 82, 'female'),
    ('GSM300339', 'C98', 'hippocampus', 82, 'female'),
    ('GSM300340', 'C98', 'postcentral gyrus', 82, 'female'),
    ('GSM300341', 'C98', 'superior frontal gyrus', 82, 'female'),
    ('GSM318840', 'C63', 'superior frontal gyrus', 66, 'male'),
    ('GSM350078', 'C15', 'superior frontal gyrus', 80, 'male'),
    ('GSM1176196', 'AD48', 'entorhinal cortex', 76, 'female'),
    ('GSM1176197', 'AD15', 'entorhinal cortex', 86, 'female'),
    ('GSM1176198', 'AD9', 'entorhinal cortex', 82, 'female'),
    ('GSM1176199', 'AD24', 'entorhinal cortex', 85, 'female'),
    ('GSM1176200', 'AD36', 'entorhinal cortex', 90, 'female'),
    ('GSM1176201', 'AD6', 'entorhinal cortex', 90, 'female'),
    ('GSM1176202', 'AD75', 'entorhinal cortex', 90, 'female'),
    ('GSM1176203', 'AD11', 'entorhinal cortex', 91, 'female'),
    ('GSM1176204', 'AD69', 'entorhinal cortex', 76, 'male'),
    ('GSM1176205', 'AD51', 'entorhinal cortex', 85, 'male'),
    ('GSM1176206', 'AD74', 'entorhinal cortex', 85, 'male'),
    ('GSM1176207', 'AD92', 'entorhinal cortex', 86, 'male'),
    ('GSM1176208', 'AD19', 'entorhinal cortex', 87, 'male'),
    ('GSM1176209', 'AD5', 'entorhinal cortex', 94, 'male'),
    ('GSM1176210', 'AD90', 'entorhinal cortex', 94, 'male'),
    ('GSM1176211', 'AD20', 'hippocampus', 60, 'female'),
    ('GSM1176212', 'AD37', 'hippocampus', 74, 'female'),
    ('GSM1176213', 'AD48', 'hippocampus', 76, 'female'),
    ('GSM1176214', 'AD16', 'hippocampus', 79, 'female'),
    ('GSM1176215', 'AD15', 'hippocampus', 86, 'female'),
    ('GSM1176216', 'AD24', 'hippocampus', 85, 'female'),
    ('GSM1176217', 'AD23', 'hippocampus', 90, 'female'),
    ('GSM1176218', 'AD36', 'hippocampus', 90, 'female'),
    ('GSM1176219', 'AD75', 'hippocampus', 90, 'female'),
    ('GSM1176220', 'AD11', 'hippocampus', 91, 'female'),
    ('GSM1176221', 'AD33', 'hippocampus', 76, 'male'),
    ('GSM1176222', 'AD69', 'hippocampus', 76, 'male'),
    ('GSM1176223', 'AD7', 'hippocampus', 79, 'male'),
    ('GSM1176224', 'AD43', 'hippocampus', 80, 'male'),
    ('GSM1176225', 'AD51', 'hippocampus', 85, 'male'),
    ('GSM1176226', 'AD92', 'hippocampus', 86, 'male'),
    ('GSM1176227', 'AD19', 'hippocampus', 87, 'male'),
    ('GSM1176228', 'AD5', 'hippocampus', 94, 'male'),
    ('GSM1176229', 'AD90', 'hippocampus', 94, 'male'),
    ('GSM1176230', 'AD20', 'postcentral gyrus', 60, 'female'),
    ('GSM1176231', 'AD37', 'postcentral gyrus', 74, 'female'),
    ('GSM1176232', 'AD48', 'postcentral gyrus', 76, 'female'),
    ('GSM1176233', 'AD15', 'postcentral gyrus', 86, 'female'),
    ('GSM1176234', 'AD13', 'postcentral gyrus', 82, 'female'),
    ('GSM1176235', 'AD9', 'postcentral gyrus', 82, 'female'),
    ('GSM1176236', 'AD24', 'postcentral gyrus', 85, 'female'),
    ('GSM1176237', 'AD57', 'postcentral gyrus', 85, 'female'),
    ('GSM1176238', 'AD23', 'postcentral gyrus', 90, 'female'),
    ('GSM1176239', 'AD36', 'postcentral gyrus', 90, 'female'),
    ('GSM1176240', 'AD6', 'postcentral gyrus', 90, 'female'),
    ('GSM1176241', 'AD75', 'postcentral gyrus', 90, 'female'),
    ('GSM1176242', 'AD11', 'postcentral gyrus', 91, 'female'),
    ('GSM1176243', 'AD70', 'postcentral gyrus', 94, 'female'),
    ('GSM1176244', 'AD64', 'postcentral gyrus', 95, 'female'),
    ('GSM1176245', 'AD33', 'postcentral gyrus', 76, 'male'),
    ('GSM1176246', 'AD69', 'postcentral gyrus', 76, 'male'),
    ('GSM1176247', 'AD7', 'postcentral gyrus', 79, 'male'),
    ('GSM1176248', 'AD51', 'postcentral gyrus', 85, 'male'),
    ('GSM1176249', 'AD74', 'postcentral gyrus', 85, 'male'),
    ('GSM1176250', 'AD92', 'postcentral gyrus', 86, 'male'),
    ('GSM1176251', 'AD19', 'postcentral gyrus', 87, 'male'),
    ('GSM1176252', 'AD44', 'postcentral gyrus', 94, 'male'),
    ('GSM1176253', 'AD5', 'postcentral gyrus', 94, 'male'),
    ('GSM1176254', 'AD90', 'postcentral gyrus', 94, 'male'),
    ('GSM1176255', 'AD37', 'superior frontal gyrus', 74, 'female'),
    ('GSM1176256', 'AD48', 'superior frontal gyrus', 76, 'female'),
    ('GSM1176257', 'AD15', 'superior frontal gyrus', 86, 'female'),
    ('GSM1176258', 'AD9', 'superior frontal gyrus', 82, 'female'),
    ('GSM1176259', 'AD24', 'superior frontal gyrus', 85, 'female'),
    ('GSM1176260', 'AD57', 'superior frontal gyrus', 85, 'female'),
    ('GSM1176261', 'AD91', 'superior frontal gyrus', 88, 'female'),
    ('GSM1176262', 'AD23', 'superior frontal gyrus', 90, 'female'),
    ('GSM1176263', 'AD36', 'superior frontal gyrus', 90, 'female'),
    ('GSM1176264', 'AD6', 'superior frontal gyrus', 90, 'female'),
    ('GSM1176265', 'AD75', 'superior frontal gyrus', 90, 'female'),
    ('GSM1176266', 'AD11', 'superior frontal gyrus', 91, 'female'),
    ('GSM1176267', 'AD70', 'superior frontal gyrus', 94, 'female'),
    ('GSM1176268', 'AD64', 'superior frontal gyrus', 95, 'female'),
    ('GSM1176269', 'AD33', 'superior frontal gyrus', 76, 'male'),
    ('GSM1176270', 'AD51', 'superior frontal gyrus', 85, 'male'),
    ('GSM1176271', 'AD74', 'superior frontal gyrus', 85, 'male'),
    ('GSM1176272', 'AD92', 'superior frontal gyrus', 86, 'male'),
    ('GSM1176273', 'AD44', 'superior frontal gyrus', 94, 'male'),
    ('GSM1176274', 'AD5', 'superior frontal gyrus', 94, 'male'),
    ('GSM1176275', 'AD90', 'superior frontal gyrus', 94, 'male'),
]
gp = '/kaggle/input/alzheimercsv/GSE48350_subjects_by_genes.csv'
lp = '/kaggle/input/labels1/GSE48350_clean_labels.csv'
if not os.path.exists(gp):
    gp = glob.glob('/kaggle/input/**/GSE48350_subjects_by_genes.csv', recursive=True)[0]
    lp = glob.glob('/kaggle/input/**/GSE48350_clean_labels.csv', recursive=True)[0]
g = pd.read_csv(gp)
g['Subject_ID'] = g['Subject_ID'].str.extract(r'(GSM\d+)')
g = g.set_index('Subject_ID')
lab = pd.read_csv(lp).set_index('GSM_ID')['GROUP']
m = pd.DataFrame(GSE48350_META, columns=['gsm', 'donor', 'region', 'age', 'sex']).set_index('gsm')
e = maybe_log2(g.T.astype(float), "GSE48350")
# symbols already collapsed by the project's preprocessing; normalise names only
names = pd.Series(e.index, index=e.index).map(lambda s: norm_symbol(s) or s)
e = e.groupby(names.values).mean()
m['dx'] = lab.reindex(m.index)
m['donor'] = 'GSE48350_' + m['donor']
COH = {"GSE48350": (e, m.assign(platform="GPL570"))}
print("GSE48350:", e.shape, m.dx.value_counts().to_dict())


In [ ]:
# ======================================================================
# GSE33000 — prefrontal cortex, Agilent/Rosetta 44K (GPL4372)
# ======================================================================
expr, meta = read_series_matrix(series_matrix("GSE33000"))
ann = platform_table("GPL4372")
e = probes_to_symbols(maybe_log2(expr, "GSE33000"), ann, entrez_col="EntrezGeneID", name="GSE33000")
dx = meta["disease status"].map({"Alzheimer's disease": "AD", "non-demented": "Control"})
m = pd.DataFrame({"donor": "GSE33000_" + meta.index,          # one array per person
                  "region": "prefrontal cortex",
                  "age": pd.to_numeric(meta["age"].str.extract(r"(\d+)")[0], errors="coerce"),
                  "sex": meta["gender"].str.lower(), "dx": dx}, index=meta.index)
COH["GSE33000"] = (e, m.assign(platform="GPL4372"))
print("GSE33000:", e.shape, m.dx.value_counts(dropna=False).to_dict())


In [ ]:
# ======================================================================
# GSE132903 — middle temporal gyrus, Illumina HT-12 v4 (GPL10558)
# ======================================================================
expr, meta = read_series_matrix(series_matrix("GSE132903"))
ann10558 = platform_table("GPL10558")
e = probes_to_symbols(maybe_log2(expr, "GSE132903"), ann10558,
                      entrez_col="Entrez_Gene_ID", symbol_col="Symbol", name="GSE132903")
agecol = [c for c in meta.columns if "age" in c][0]
m = pd.DataFrame({"donor": "GSE132903_" + meta.index,          # one array per person
                  "region": "middle temporal gyrus",
                  "age": pd.to_numeric(meta[agecol].str.extract(r"(\d+)")[0], errors="coerce"),  # '90+' -> 90
                  "sex": meta["sex"].str.lower(),
                  "dx": meta["diagnosis"].map({"AD": "AD", "ND": "Control"})}, index=meta.index)
COH["GSE132903"] = (e, m.assign(platform="GPL10558"))
print("GSE132903:", e.shape, m.dx.value_counts(dropna=False).to_dict())


In [ ]:
# ======================================================================
# GSE118553 — temporal / entorhinal / frontal cortex (+cerebellum), GPL10558
# ======================================================================
expr, meta = read_series_matrix(series_matrix("GSE118553"))
e = probes_to_symbols(maybe_log2(expr, "GSE118553"), ann10558,
                      entrez_col="Entrez_Gene_ID", symbol_col="Symbol", name="GSE118553")
region = meta["tissue"].str.replace("_", " ").str.lower()
dx = meta["disease state"].map({"AD": "AD", "control": "Control"})   # AsymAD -> NaN (dropped)
dx[region == "cerebellum"] = np.nan                                   # not cortex
m = pd.DataFrame({"donor": "GSE118553_" + meta["individual"].str.replace(r"\s+", "", regex=True),
                  "region": region,
                  "age": pd.to_numeric(meta["age"], errors="coerce"),
                  "sex": meta["gender"].str.lower(), "dx": dx}, index=meta.index)
COH["GSE118553"] = (e, m.assign(platform="GPL10558"))
print("GSE118553:", e.shape, m.dx.value_counts(dropna=False).to_dict())


In [ ]:
# ======================================================================
# GSE84422 — Mount Sinai Brain Bank, 17 regions on GPL96 (Affymetrix U133A)
# ======================================================================
expr, meta = read_series_matrix(series_matrix("GSE84422", "-GPL96"))
ann96 = platform_table("GPL96")
e = probes_to_symbols(maybe_log2(expr, "GSE84422"), ann96,
                      entrez_col="ENTREZ_GENE_ID", symbol_col="Gene Symbol", name="GSE84422")
cat = meta["neuropathological category"]
dx = cat.map({"definite AD": "AD", "Probable AD": "AD", "Normal": "Control"})   # Possible AD dropped
region = meta["brain region"].str.lower()
dx[region.isin(["caudate nucleus", "putamen"])] = np.nan                       # subcortical
m = pd.DataFrame({"donor": "GSE84422_" + meta["subject id"].astype(str),
                  "region": region,
                  "age": pd.to_numeric(meta["age"], errors="coerce"),
                  "sex": meta["sex"].str.lower(), "dx": dx}, index=meta.index)
COH["GSE84422"] = (e, m.assign(platform="GPL96"))
print("GSE84422:", e.shape, m.dx.value_counts(dropna=False).to_dict())


In [ ]:
# ======================================================================
# Filter, intersect genes, z-score within cohort, merge
# ======================================================================
rows, mats, metas = [], [], []
for name, (e, m) in COH.items():
    m = m.copy(); m["cohort"] = name
    m = m[m.index.isin(e.columns)]
    n0 = len(m)
    m = m[m.dx.isin(["AD", "Control"])]
    n_dx = len(m)
    m = m[m.age >= AGE_MIN]
    rows.append({"cohort": name, "platform": m.platform.iloc[0], "arrays_in_geo": n0,
                 "after_diagnosis_filter": n_dx, "after_age_filter": len(m),
                 "donors": m.donor.nunique(),
                 "AD_arrays": int((m.dx == "AD").sum()), "control_arrays": int((m.dx == "Control").sum()),
                 "AD_donors": m[m.dx == "AD"].donor.nunique(),
                 "control_donors": m[m.dx == "Control"].donor.nunique(),
                 "age_median_AD": m[m.dx == "AD"].drop_duplicates("donor").age.median(),
                 "age_median_control": m[m.dx == "Control"].drop_duplicates("donor").age.median(),
                 "regions": m.region.nunique(), "genes_measured": e.shape[0]})
    mats.append(e[m.index]); metas.append(m)

common = sorted(set.intersection(*[set(x.index) for x in mats]))
cov = pd.DataFrame({"gene": sorted(set.union(*[set(x.index) for x in mats]))})
for name, x in zip(COH, mats):
    cov[name] = cov.gene.isin(x.index)
cov["in_all"] = cov.gene.isin(common)
cov.to_csv(f"{OUT}/gene_coverage.csv", index=False)
print(f"genes measured in every cohort: {len(common):,}")

zs = []
for x in mats:
    x = x.loc[common]
    x = x.T.fillna(x.T.mean())                   # rare missing values -> cohort gene mean
    sd = x.std(ddof=0).replace(0, 1)
    zs.append((x - x.mean()) / sd)                # per-cohort, per-gene z-score (label-free)
X = pd.concat(zs).astype(np.float32)
M = pd.concat(metas).loc[X.index]
assert not X.index.duplicated().any()
keep_genes = X.columns[X.std() > 0]
X = X[keep_genes]

table = pd.DataFrame(rows)
total = table.sum(numeric_only=True); total["cohort"] = "TOTAL"
total["age_median_AD"] = M[M.dx == "AD"].drop_duplicates("donor").age.median()
total["age_median_control"] = M[M.dx == "Control"].drop_duplicates("donor").age.median()
total["genes_measured"] = X.shape[1]
table = pd.concat([table, total.to_frame().T], ignore_index=True)
table.to_csv(f"{OUT}/cohort_table.csv", index=False)
print(table.to_string(index=False))

X.index.name = "Subject_ID"
X.round(5).to_csv(f"{OUT}/merged_expression.csv.gz", compression="gzip")
pd.DataFrame({"GSM_ID": M.index, "GROUP": M.dx.values}).to_csv(f"{OUT}/merged_labels.csv", index=False)
M.rename_axis("Subject_ID")[["cohort", "platform", "donor", "region", "age", "sex", "dx"]].to_csv(
    f"{OUT}/merged_metadata.csv")
print(f"\nmerged: {X.shape[0]:,} arrays x {X.shape[1]:,} genes, {M.donor.nunique()} donors")


In [ ]:
# ======================================================================
# QC: does cohort still dominate after per-cohort z-scoring?
# ======================================================================
from sklearn.decomposition import PCA
raw = pd.concat([x.loc[common].T for x in mats]).loc[X.index]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (title, data, col) in zip(axes, [("before: raw values, by cohort", raw.fillna(raw.mean()), "cohort"),
                                         ("after: z-scored per cohort, by cohort", X, "cohort"),
                                         ("after: z-scored per cohort, by diagnosis", X, "dx")]):
    pc = PCA(2, random_state=0).fit_transform(data.values)
    for k in sorted(M[col].unique()):
        sel = (M[col] == k).values
        ax.scatter(pc[sel, 0], pc[sel, 1], s=6, alpha=0.6, label=k)
    ax.set_title(title, fontsize=10); ax.legend(fontsize=7, markerscale=2, frameon=False)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT}/qc_pca_merge.png", dpi=200); plt.show()
